In [ ]:
# code1_gnn_plus_kge.py
# 双支路：TransE + GraphSAGE → 拼接 → TransH

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import GraphSAINTNodeSampler
from torch_geometric.utils import negative_sampling
import random
import numpy as np
import os
from tqdm import tqdm
import faiss
import zipfile

# ==================== 固定随机种子 ====================
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# ==================== 配置 ====================
scheme_type = "s9_in_ref"
BASE_DIR = "/mnt/d/forCoding_data/Tianchi_EcommerceKG"
TRAIN_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_train.tsv"
DEV_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_dev.tsv"
TEST_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_test.tsv"
OUTPUT_FILE_PATH = f"{BASE_DIR}/preprocessedData/OpenBG500_test.tsv"
MODEL_DIR = f"{BASE_DIR}/trained_models/{scheme_type}"
os.makedirs(MODEL_DIR, exist_ok=True)

TRAINED_MODEL_PATHS = {
    'TransE': f"{MODEL_DIR}/transE.pth",
    'GraphSAGE': f"{MODEL_DIR}/graphsage.pth",
    'TransH': f"{MODEL_DIR}/transH.pth"
}

# 超参数
EMBEDDING_DIM = 100
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 5
BATCH_SIZE = 256
NEGATIVE_SAMPLES = 10
MAX_LINES = None
FORCE_RETRAIN = False

# ==================== 数据集 ====================
class KnowledgeGraphDataset(torch.utils.data.Dataset):
    def __init__(self, file_path, is_test=False, max_lines=None, is_train=False):
        self.triples = []
        self.is_train = is_train
        self._load_data(file_path, is_test, max_lines)
    def _load_data(self, file_path, is_test, max_lines):
        print(f"加载数据: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if max_lines: lines = lines[:max_lines]
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 3:
                    h, r, t = parts
                    self.triples.append((h, r, t))
                elif is_test and len(parts) == 2:
                    h, r = parts
                    self.triples.append((h, r, "<UNK>"))
        print(f"共加载 {len(self.triples)} 个三元组")
    def __len__(self): return len(self.triples)
    def __getitem__(self, idx): return self.triples[idx]

def collate_fn(batch):
    h_list, r_list, t_list = zip(*batch)
    return list(h_list), list(r_list), list(t_list)

# ==================== 映射器 ====================
class EntityRelationMapper:
    def __init__(self):
        self.entity_to_id = {}
        self.id_to_entity = {}
        self.relation_to_id = {}
        self.id_to_relation = {}
        self.entity_count = 0
        self.relation_count = 0
        self.all_train_triples = []
    def build_mappings(self, *datasets):
        entities = set()
        relations = set()
        for dataset in datasets:
            for h, r, t in dataset.triples:
                entities.add(h); entities.add(t); relations.add(r)
                if dataset.is_train: self.all_train_triples.append((h, r, t))
        for e in sorted(entities):
            self.entity_to_id[e] = self.entity_count
            self.id_to_entity[self.entity_count] = e
            self.entity_count += 1
        for r in sorted(relations):
            self.relation_to_id[r] = self.relation_count
            self.id_to_relation[self.relation_count] = r
            self.relation_count += 1

# ==================== TransE ====================
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)
    def forward(self, h, r, t):
        return torch.norm(self.E(h) + self.R(r) - self.E(t), p=1, dim=1)
    def get_query_embedding(self, h, r):
        return self.E(h) + self.R(r)
    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== GraphSAGE ====================
class GraphSAGE(nn.Module):
    def __init__(self, num_entities, dim):
        super().__init__()
        self.embedding = nn.Embedding(num_entities, dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        self.conv1 = SAGEConv(dim, dim, aggr='mean')
        self.conv2 = SAGEConv(dim, dim, aggr='mean')
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        x = self.embedding(x)
        x = self.dropout(torch.relu(self.conv1(x, edge_index)))
        x = self.conv2(x, edge_index)
        return x

# ==================== TransH ====================
class TransH(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        self.W = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)
        nn.init.xavier_uniform_(self.W.weight)
    def project(self, emb, w):
        norm_w = torch.nn.functional.normalize(w, p=2, dim=1)
        scale = torch.sum(emb * norm_w, dim=1, keepdim=True)
        return emb - scale * norm_w
    def forward(self, h, r, t):
        h_emb = self.E(h); t_emb = self.E(t); r_vec = self.R(r); W = self.W(r)
        h_proj = self.project(h_emb, W); t_proj = self.project(t_emb, W)
        return torch.norm(h_proj + r_vec - t_proj, p=1, dim=1)
    def get_query_embedding(self, h, r):
        h_emb = self.E(h); r_vec = self.R(r); W = self.W(r)
        h_proj = self.project(h_emb, W)
        return h_proj + r_vec
    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== Train TransE ====================
def train_TransE(train_dataset, mapper, device):
    if os.path.exists(TRAINED_MODEL_PATHS['TransE']) and not FORCE_RETRAIN:
        print("[TransE] 模型已存在，跳过训练")
        ckpt = torch.load(TRAINED_MODEL_PATHS['TransE'], map_location='cpu')
        return ckpt['model_state_dict']['E.weight'].to(device)

    print("🚀 开始训练 TransE...")
    model = TransE(mapper.entity_count, mapper.relation_count, EMBEDDING_DIM).to(device)
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    model.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"TransE Epoch {epoch+1}")
        for h_list, r_list, t_list in pbar:
            h = torch.tensor([mapper.entity_to_id[h] for h in h_list], device=device)
            r = torch.tensor([mapper.relation_to_id[r] for r in r_list], device=device)
            t = torch.tensor([mapper.entity_to_id[t] for t in t_list], device=device)
            neg_t = torch.randint(0, mapper.entity_count, (len(h), NEGATIVE_SAMPLES), device=device)
            pos_score = model(h, r, t)
            neg_score = model(h.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              r.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              neg_t.reshape(-1)).reshape(-1, NEGATIVE_SAMPLES)
            loss = torch.mean(torch.relu(pos_score.unsqueeze(1) - neg_score + 1.0))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            model.normalize_entities()
            epoch_loss += loss.item(); pbar.set_postfix(loss=loss.item())
        print(f"[TransE] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'entity_count': mapper.entity_count,
        'relation_count': mapper.relation_count,
        'embedding_dim': EMBEDDING_DIM,
        'entity_to_id': mapper.entity_to_id,
        'relation_to_id': mapper.relation_to_id,
    }, TRAINED_MODEL_PATHS['TransE'])
    print("[TransE] 模型已保存")
    return model.E.weight.data.detach()

# ==================== Train Second TransE ====================
def train_TransE_2(train_dataset, mapper, device):
    path = TRAINED_MODEL_PATHS['TransE'].replace('.pth', '_v2.pth')
    if os.path.exists(path) and not FORCE_RETRAIN:
        print("[TransE_v2] 模型已存在，跳过训练")
        ckpt = torch.load(path, map_location='cpu')
        return ckpt['model_state_dict']['E.weight'].to(device)

    print("🚀 开始训练 TransE_v2...")
    model = TransE(mapper.entity_count, mapper.relation_count, EMBEDDING_DIM).to(device)
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    model.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"TransE_v2 Epoch {epoch+1}")
        for h_list, r_list, t_list in pbar:
            h = torch.tensor([mapper.entity_to_id[h] for h in h_list], device=device)
            r = torch.tensor([mapper.relation_to_id[r] for r in r_list], device=device)
            t = torch.tensor([mapper.entity_to_id[t] for t in t_list], device=device)
            neg_t = torch.randint(0, mapper.entity_count, (len(h), NEGATIVE_SAMPLES), device=device)
            pos_score = model(h, r, t)
            neg_score = model(h.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              r.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              neg_t.reshape(-1)).reshape(-1, NEGATIVE_SAMPLES)
            loss = torch.mean(torch.relu(pos_score.unsqueeze(1) - neg_score + 1.0))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            model.normalize_entities()
            epoch_loss += loss.item(); pbar.set_postfix(loss=loss.item())
        print(f"[TransE_v2] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'entity_count': mapper.entity_count,
        'relation_count': mapper.relation_count,
        'embedding_dim': EMBEDDING_DIM,
        'entity_to_id': mapper.entity_to_id,
        'relation_to_id': mapper.relation_to_id,
    }, path)
    print("[TransE_v2] 模型已保存")
    return model.E.weight.data.detach()

# ==================== Train TransH with Concat Init ====================
def train_TransH_with_concat(train_dataset, mapper, device, e1_weight, e2_weight):
        
    assert e1_weight.shape[1] == e2_weight.shape[1] == EMBEDDING_DIM
    fused_dim = EMBEDDING_DIM * 2
    print(f"🔧 使用拼接初始化 TransH，输入维度: {fused_dim}")

    model = TransH(mapper.entity_count, mapper.relation_count, fused_dim).to(device)
    combined_E = torch.cat([e1_weight, e2_weight], dim=1)  # [N, 200]
    model.E.weight.data.copy_(combined_E)

    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    model.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"TransH Epoch {epoch+1}")
        for h_list, r_list, t_list in pbar:
            h = torch.tensor([mapper.entity_to_id[h] for h in h_list], device=device)
            r = torch.tensor([mapper.relation_to_id[r] for r in r_list], device=device)
            t = torch.tensor([mapper.entity_to_id[t] for t in t_list], device=device)
            neg_t = torch.randint(0, mapper.entity_count, (len(h), NEGATIVE_SAMPLES), device=device)
            pos_score = model(h, r, t)
            neg_score = model(h.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              r.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              neg_t.reshape(-1)).reshape(-1, NEGATIVE_SAMPLES)
            loss = torch.mean(torch.relu(pos_score.unsqueeze(1) - neg_score + 1.0))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            model.normalize_entities()
            epoch_loss += loss.item(); pbar.set_postfix(loss=loss.item())
        print(f"[TransH] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'entity_count': mapper.entity_count,
        'relation_count': mapper.relation_count,
        'embedding_dim': fused_dim,
        'entity_to_id': mapper.entity_to_id,
        'relation_to_id': mapper.relation_to_id,
    }, TRAINED_MODEL_PATHS['TransH'])
    print("[TransH] 模型已保存")
    return model

# ==================== Evaluate & Predict ====================
def evaluate_and_predict(model, dev_data, test_data, mapper, device):
    entity_emb = model.E.weight.data.cpu().numpy()
    index = faiss.IndexFlatL2(entity_emb.shape[1])
    index.add(entity_emb)

    # Dev 评估
    hits_at = {1:0, 3:0, 10:0}; mrr = 0; count = 0
    for h, r, t in tqdm(dev_data.triples, desc="Evaluating"):
        try:
            h_id = torch.tensor([mapper.entity_to_id[h]], device=device)
            r_id = torch.tensor([mapper.relation_to_id[r]], device=device)
            t_id = mapper.entity_to_id[t]
        except KeyError: continue
        query = model.get_query_embedding(h_id, r_id).cpu().detach().numpy()
        _, indices = index.search(query, 1000)
        pred_ids = indices[0]
        rank = np.where(pred_ids == t_id)[0]
        final_rank = rank[0] + 1 if len(rank) > 0 else 10000
        for k in hits_at: hits_at[k] += 1 if final_rank <= k else 0
        mrr += 1.0 / final_rank; count += 1

    for k in hits_at: hits_at[k] /= count
    mrr /= count
    print(f"HITS@1: {hits_at[1]:.4f}, HITS@3: {hits_at[3]:.4f}, HITS@10: {hits_at[10]:.4f}, MRR: {mrr:.4f}")

    # Test 预测
    results = []
    for h, r, _ in tqdm(test_data.triples, desc="Predict"):
        try:
            h_id = torch.tensor([mapper.entity_to_id[h]], device=device)
            r_id = torch.tensor([mapper.relation_to_id[r]], device=device)
        except KeyError:
            preds = [h] * 10
            results.append('\t'.join([h, r] + preds))
            continue
        q = model.get_query_embedding(h_id, r_id).cpu().detach().numpy()
        _, indices = index.search(q, 10)
        preds = [mapper.id_to_entity[i] for i in indices[0]]
        results.append('\t'.join([h, r] + preds))

    os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)
    with open(OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(results) + '\n')

    zip_path = OUTPUT_FILE_PATH.replace(".tsv", "") + f"__{scheme_type}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_FILE_PATH, arcname=os.path.basename(OUTPUT_FILE_PATH))
        
    print(f"✅ 预测结果已保存: {OUTPUT_FILE_PATH}")

# ==================== 主函数 ====================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch, 'mps') and torch.backends.mps.is_available() else 'cpu')
    print(f"🚀 使用设备: {device}")

    train_data = KnowledgeGraphDataset(TRAIN_FILE_PATH, max_lines=MAX_LINES, is_train=True)
    dev_data = KnowledgeGraphDataset(DEV_FILE_PATH, is_test=False, is_train=False)
    test_data = KnowledgeGraphDataset(TEST_FILE_PATH, is_test=True, is_train=False)
    mapper = EntityRelationMapper()
    mapper.build_mappings(train_data, dev_data, test_data)
    print(f"实体数: {mapper.entity_count}, 关系数: {mapper.relation_count}")

    # 支路1: TransE
    transE1_E = train_TransE(train_data, mapper, device)
    # 支路2: TransE_v2
    transE2_E = train_TransE_2(train_data, mapper, device)
    # TransH 暖启动
    transH_model = train_TransH_with_concat(train_data, mapper, device, transE1_E, transE2_E)
    # 评估与预测
    evaluate_and_predict(transH_model, dev_data, test_data, mapper, device)

if __name__ == "__main__":
    main()